In [9]:
import pandas as pd
import numpy as np
np.int=int
from astroquery.simbad import Simbad
from astropy.coordinates import SkyCoord
import astropy.units as u
import sfdmap

import os
import time
import extinction
from jedi.inference.compiled import create_from_access_path

In [10]:
CFA_PARAM_FILE = 'cfasnIa_param.dat'
OUTPUT_FILE = 'supernova_extinction_data.csv'

# Initialize SIMBAD and sfdmap
Simbad.add_votable_fields('ra', 'dec') # Request RA and Dec in degrees
sfd = sfdmap.SFDMap('/Users/pxm588@student.bham.ac.uk/Desktop/snid/all_raw_cfa_spectra/sfddata')


In [11]:
def get_sn_names(file_path):
    """
    Parses the cfasnIa_param.dat file to extract unique supernova names.
    """
    sn_names = set()
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue

            # SN name is the first column
            parts = line.split()
            if parts:
                sn_names.add(parts[0])
    return sorted(list(sn_names))

In [12]:
testname = get_sn_names(CFA_PARAM_FILE)[0]

In [13]:
print(testname)

1993ac


In [14]:
def query_simbad_and_sfd(sn_name):
    """
    Queries SIMBAD for supernova coordinates and then uses sfdmap for galactic extinction.
    """
    ra, dec, ebv = np.nan, np.nan, np.nan
    candidate_simbad_ids = []
    if sn_name.startswith('SNF'):
        candidate_simbad_ids.append(sn_name)
    else:
        candidate_simbad_ids.append('sn'+ sn_name)

    for simbad_id_to_try in candidate_simbad_ids:
        try:
            result_table = Simbad.query_object(simbad_id_to_try)
            if result_table is not None and len(result_table) > 0:
                ra_deg = result_table['ra'][0]
                dec_deg = result_table['dec'][0]

                coords = SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame='icrs')
                ra, dec = coords.ra.deg, coords.dec.deg

                ebv = sfd.ebv(ra, dec, scale_by_r_v=False)

                print(f"Found {sn_name} using SIMBAD ID '{simbad_id_to_try}': RA={ra:.4f}, Dec={dec:.4f}, E(B-V)={ebv:.4f}")
                return ra, dec, ebv # Success!
            else:
                print(f"Could not find {sn_name} using SIMBAD ID '{simbad_id_to_try}'.")
        except Exception as e:
            print(f"Error querying SIMBAD for {sn_name} using SIMBAD ID '{simbad_id_to_try}': {e}")

    return ra, dec, ebv # If nothing found after all attempts


In [15]:
test_results = query_simbad_and_sfd(testname)

Found 1993ac using SIMBAD ID 'sn1993ac': RA=86.5982, Dec=63.3686, E(B-V)=0.1390


In [16]:
(print(test_results))
test_ebv = sfd.ebv(test_results[0], test_results[1])

(86.59825, 63.36863888888889, 0.13901358988422918)


NameError: name 'm' is not defined